# Results Aggregator

Loads every per-`(model, wrapper, ticker, regime)` interval CSV under `results/` and produces the paper's tables + the headline Cov_low vs Cov_high scatter.

Tolerates partial data — will produce tables for whichever (model, wrapper) combinations have been run. Rerun after dropping more CSVs into `results/`.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / "requirements.txt").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis._shared import load_all_intervals, CRISIS_WINDOWS
from src.metrics.metrics import Metrics
from src.metrics.prediction_cols import PredictionCols

ALPHA = 0.10  # matches Metrics.md

## Load and inspect saved intervals

In [ ]:
raw = load_all_intervals()
raw["date"] = pd.to_datetime(raw["date"])
# Pandas infers int dtype when all regimes happen to look numeric (e.g. "2020").
# Force string so CRISIS_WINDOWS lookup works.
raw["regime"] = raw["regime"].astype(str)

print(f"{len(raw):,} rows across {raw.groupby(['model','wrapper','ticker','regime']).ngroups} (model, wrapper, ticker, regime) combos")

# What's present / missing?
avail = raw.groupby(["model", "wrapper"]).size().rename("rows").reset_index()
avail_pivot = avail.pivot(index="model", columns="wrapper", values="rows").fillna(0).astype(int)
print("\nRows per (model, wrapper):")
avail_pivot

## Per-slice metric computation

`metrics_for_slice` takes one `(model, wrapper, ticker, regime)` slice, renames columns to the PredictionCols schema, and returns every scalar the tables and figures need.

Vol-threshold caveat: the `Metrics` facade recomputes the 75th-percentile threshold on each slice. Since (model, wrapper) pairs on the same (ticker, regime) share identical realized returns, cross-model comparisons *within* a (ticker, regime) are valid. Cross-ticker / cross-regime comparisons implicitly use different thresholds — fine because each row of the reported tables is already stratified by ticker and regime.

In [ ]:
def metrics_for_slice(slice_df: pd.DataFrame, regime: str) -> dict:
    """Compute every scalar needed for the paper tables for one (model, wrapper, ticker, regime) slice."""
    s = slice_df.rename(columns={
        "r": PredictionCols.LOG_RETURNS,
        "l": PredictionCols.LOWER_BOUND,
        "u": PredictionCols.UPPER_BOUND,
    }).set_index("date").sort_index()

    # Drop infinite-width rows from the metric computation (they signal a
    # calibration blow-up; keep a tally for the diagnostic column).
    widths = s[PredictionCols.UPPER_BOUND] - s[PredictionCols.LOWER_BOUND]
    n_inf = int(np.isinf(widths).sum())
    s_finite = s[~np.isinf(widths)]
    if len(s_finite) == 0:
        return {"n": 0, "inf_days": n_inf}

    m = Metrics(s_finite, alpha=ALPHA)
    cov_m = m.get_coverage_metrics()
    win_m = m.get_winkler_metrics()
    int_m = m.get_interval_metrics()
    ada_m = m.get_adaptation_metrics()

    marginal_cov = float(cov_m.get_marginal_coverage())
    cov_high, cov_low, gap = cov_m.get_high_low_vol_coverage()
    w_overall, w_high, w_low = win_m.get_mean_scores()
    width_overall, width_high, width_low = int_m.get_mean_widths()

    # Crisis-window subset
    cfg = CRISIS_WINDOWS[regime]
    onset, end = cfg["onset"], cfg["end"]
    crisis_cov  = float(cov_m.get_period_coverage(onset, end))
    crisis_win  = float(win_m.get_period_score(onset, end))
    crisis_wid  = float(int_m.get_period_width(onset, end))
    T_rec, max_drawdown, recovered = ada_m.compute_adaptation_speed(
        onset, end, threshold=0.85, persistence=5, window=20
    )
    # Robustness variant (Metrics.md requires reporting)
    T_rec_robust, _, _ = ada_m.compute_adaptation_speed(
        onset, end, threshold=0.80, persistence=1, window=20
    )

    return {
        "n": len(s_finite),
        "inf_days": n_inf,
        "marginal_cov": marginal_cov,
        "cov_high": float(cov_high),
        "cov_low": float(cov_low),
        "gap": float(gap),
        "w_overall": float(w_overall),
        "w_high": float(w_high),
        "w_low": float(w_low),
        "width_overall": float(width_overall),
        "width_high": float(width_high),
        "width_low": float(width_low),
        "crisis_cov": crisis_cov,
        "crisis_win": crisis_win,
        "crisis_wid": crisis_wid,
        "t_recovery": int(T_rec),
        "max_drawdown_cov": float(max_drawdown) if max_drawdown is not None else float("nan"),
        "recovered": bool(recovered),
        "t_recovery_robust": int(T_rec_robust),
    }

In [ ]:
long = []
for (model, wrapper, ticker, regime), g in raw.groupby(["model", "wrapper", "ticker", "regime"]):
    row = {"model": model, "wrapper": wrapper, "ticker": ticker, "regime": regime}
    row.update(metrics_for_slice(g, regime))
    long.append(row)
long = pd.DataFrame(long)
print(f"Computed metrics for {len(long)} slices")
long.head()

## T1 — Main results table

Average across (ticker, regime) per (model, wrapper). This is the headline number most reviewers look at first.

In [ ]:
T1 = long.groupby(["model", "wrapper"]).agg(
    marginal_cov=("marginal_cov", "mean"),
    cov_high=("cov_high", "mean"),
    cov_low=("cov_low", "mean"),
    gap=("gap", "mean"),
    width=("width_overall", "mean"),
    width_high=("width_high", "mean"),
    winkler=("w_overall", "mean"),
    winkler_high=("w_high", "mean"),
    inf_days=("inf_days", "sum"),
).round(4)
T1

## T2 — Per-crisis tables

Breaks T1 apart by regime so 2008, 2020, and 2022 can be inspected individually. Coverage numbers are crisis-window-only (the `crisis_*` columns from the long table).

In [ ]:
for regime in sorted(long["regime"].unique()):
    print(f"=== {regime} ===")
    subset = long[long["regime"] == regime]
    T2 = subset.groupby(["model", "wrapper"]).agg(
        crisis_cov=("crisis_cov", "mean"),
        crisis_width=("crisis_wid", "mean"),
        crisis_winkler=("crisis_win", "mean"),
    ).round(4)
    display(T2)

## T3 — Adaptation speed table

Days until rolling 20-day coverage recovers to ≥ 0.85 for 5 consecutive days after crisis onset, plus the worst rolling-coverage reading before recovery.

In [ ]:
for regime in sorted(long["regime"].unique()):
    print(f"=== {regime} ===")
    subset = long[long["regime"] == regime]
    T3 = subset.groupby(["model", "wrapper"]).agg(
        t_recovery=("t_recovery", "mean"),
        max_drawdown_cov=("max_drawdown_cov", "mean"),
        recovered_frac=("recovered", "mean"),
        t_recovery_robust=("t_recovery_robust", "mean"),
    ).round(2)
    display(T3)

## T4 — TCP reproducibility check

Aich et al. 2026 report TCP-RM on S&P 500 with marginal coverage ≈ 0.9530 and mean width ≈ 5.22 (Table 1, α=0.05). Our run uses α=0.10 on individual-stock log returns so the numbers aren't directly comparable, but we expect TCP/TCP-RM to hit near-nominal (≈0.90) with wider bands than ACI. Report the per-(asset, regime) numbers explicitly so the comparison is traceable.

In [ ]:
T4 = (
    long[long["wrapper"].isin(["tcp", "tcp_rm"])]
    .sort_values(["regime", "ticker", "model", "wrapper"])
    [["regime", "ticker", "model", "wrapper", "marginal_cov", "width_overall", "w_overall"]]
    .reset_index(drop=True)
    .round(4)
)
print("TCP / TCP-RM results (if any have been run):")
T4

## Figure — Cov_low vs Cov_high scatter

Per Metrics.md §3, this is the headline figure: one dot per (model, wrapper), coordinates averaged across (ticker, regime). Points on the y=x diagonal have consistent coverage across volatility regimes; points below under-cover during high-vol.

In [ ]:
scatter_df = long.groupby(["model", "wrapper"]).agg(
    cov_low=("cov_low", "mean"),
    cov_high=("cov_high", "mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 7))
wrapper_colors = {
    "split": "#d62728", "aci": "#2ca02c", "enbpi": "#9467bd",
    "agaci": "#1f77b4", "tcp": "#ff7f0e", "tcp_rm": "#e377c2",
    "native": "#7f7f7f",
}
model_markers = {
    "linear_ar": "o", "hist_binning": "s", "tree_based": "^",
    "lstm": "D", "transformer": "P", "fhs": "X", "garch": "*",
}

for _, row in scatter_df.iterrows():
    ax.scatter(
        row["cov_low"], row["cov_high"],
        color=wrapper_colors.get(row["wrapper"], "black"),
        marker=model_markers.get(row["model"], "o"),
        s=110, edgecolor="black", linewidth=0.5,
        label=f"{row['model']} / {row['wrapper']}",
    )

lims = [0.55, 1.0]
ax.plot(lims, lims, "k--", alpha=0.5, label="equal coverage (y=x)")
ax.axhline(0.90, color="gray", linestyle=":", alpha=0.4)
ax.axvline(0.90, color="gray", linestyle=":", alpha=0.4)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Coverage: Low volatility days")
ax.set_ylabel("Coverage: High volatility days")
ax.set_title("Conditional coverage by volatility regime")
ax.legend(fontsize=7, loc="lower right", ncols=2)
fig.tight_layout()
plt.show()

## Figure — Rolling 20-day coverage per crisis

Plots rolling-20-day coverage over time for a representative ticker per crisis, with the exogenous crisis window shaded. Every (model, wrapper) gets its own line — reveals *when* coverage collapses and recovers, not just aggregate numbers.

In [ ]:
representative = {"2008": "AIG", "2020": "MCD", "2022": "SNAP"}

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=False)
for ax, (regime, ticker) in zip(axes, representative.items()):
    onset, end = CRISIS_WINDOWS[regime]["onset"], CRISIS_WINDOWS[regime]["end"]
    slice_all = raw[(raw["regime"] == regime) & (raw["ticker"] == ticker)]
    if slice_all.empty:
        ax.set_title(f"{regime} x {ticker} — no data")
        continue
    for (model, wrapper), g in slice_all.groupby(["model", "wrapper"]):
        g = g.sort_values("date")
        covered = ((g["r"] >= g["l"]) & (g["r"] <= g["u"])).astype(float)
        rolling = covered.rolling(20).mean()
        ax.plot(g["date"], rolling, linewidth=0.9, alpha=0.8, label=f"{model}/{wrapper}")
    ax.axvspan(onset, end, alpha=0.15, color="red")
    ax.axhline(0.90, color="black", linewidth=0.8, linestyle=":", alpha=0.6)
    ax.axhline(0.85, color="gray", linewidth=0.6, linestyle=":", alpha=0.5)
    ax.set_ylabel("Rolling 20-day coverage")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{regime} — {ticker}")
    ax.legend(fontsize=7, loc="lower right", ncols=2)
fig.tight_layout()
plt.show()

## Persist the long-form metrics

Write the row-per-slice metrics table to `results/metrics_long.csv` so the paper writeup can cite exact numbers without re-running this notebook.

In [ ]:
from src.analysis._shared import RESULTS_DIR
out_path = Path(RESULTS_DIR) / "metrics_long.csv"
long.to_csv(out_path, index=False)
print(f"Wrote {len(long)} rows -> {out_path}")